# Explainability of the models

This notebook utilizes the functions defined in `utils/explainability.py` to generate LIME and SHAP explanations where applicable. By modularizing these tools, we ensure a consistent interpretability framework across different models, allowing for both global feature importance analysis and local instance-level insights.

The analysis is structured into four distinct experimental phases, we will explain only the three most significant models for each feature typology:

- **Initial Features**: This section evaluates the model using baseline data generated through hashing and other fundamental preprocessing techniques. It serves as our performance benchmark.

- **128-Dimensional Textual Embeddings**: This phase leverages high-dimensional vector representations to capture deeper semantic meaning from the text, testing whether increased complexity leads to higher predictive accuracy.

- **Graph Features**: Rather than focusing on content, this section analyzes the structural properties of the citation network (such as connectivity and node relationships) to identify patterns in how papers are linked.

- **Mix features**: The final stage combines the most significant features identified from the previous three best models. The goal is to determine if the synergy between text, structure, and metadata yields a more robust and accurate classifier.

In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from utils.explainability import lime_explainer, shap_tree_explainer

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
EXPLANATION_SAMPLE_SIZE = 5_000


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "utils").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Project root not found: miss folders utils/ and data/.")


PROJECT_ROOT = find_project_root()
TEXTUAL_FEATURES_PATH = PROJECT_ROOT / "data" / "textual_features"
GRAPH_FEATURES_PATH = PROJECT_ROOT / "data" / "graph_features"
INITIAL_FEATURES_PATH = PROJECT_ROOT / "data" / "initial_features"
MODELS_ROOT = PROJECT_ROOT / "MODELZOO"

TEXTUAL_64_PATH = TEXTUAL_FEATURES_PATH / "textual_embeddings_64.parquet"
TEXTUAL_128_PATH = TEXTUAL_FEATURES_PATH / "textual_embeddings_128.parquet"

MODEL_DIRS = {
    "embedding_64": MODELS_ROOT / "textual_embeddings_64",
    "embedding_128": MODELS_ROOT / "textual_embeddings_128",
    "graph_features": MODELS_ROOT / "graph_features",
    "initial_features": MODELS_ROOT / "initial_features"
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Textual features: {TEXTUAL_FEATURES_PATH}")
print(f"Graph features: {GRAPH_FEATURES_PATH}")
print(f"Initial features: {INITIAL_FEATURES_PATH}")
print(f"Models: {MODELS_ROOT}")

In [ ]:
def load_all_models(models_dir: Path, selected_model: str = None) -> dict[str, object]:
    """Loads all .pkl/.joblib models present in a folder."""
    if not models_dir.exists():
        print(f"Models folder not found: {models_dir}")
        return {}

    model_files = sorted(
        p for p in models_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in {".pkl", ".joblib"}
    )

    models = {}
    for model_path in model_files:
        try:
            models[model_path.stem] = joblib.load(model_path)
        except Exception as exc:
            print(f"Unable to load {model_path.name}: {exc}")
            

    if selected_model:
        model = {name: model for name, model in models.items() if selected_model in name}
        print(f"Selected model: {model}")
        return model

    print(f"Models loaded from {models_dir.name}: {len(models)}")
    for name in models:
        print(f" - {name}")
    return models

def split_train_test(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "split" in df.columns:
        split = df["split"].astype(str).str.lower()
        train_df = df[split.isin(["train", "training"])].copy()
        test_df = df[split.isin(["test", "validation", "val"])].copy()
        if len(train_df) and len(test_df):
            return train_df, test_df

    cut = int(len(df) * 0.8)
    return df.iloc[:cut].copy(), df.iloc[cut:].copy()


def load_textual_split(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_parquet(path)
    train_df, test_df = split_train_test(df)
    print(f"{path.name}: train={train_df.shape}, test={test_df.shape}")
    return train_df, test_df


def load_graph_split(graph_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_parquet(graph_dir / "train.parquet")
    test_df = pd.read_parquet(graph_dir / "test.parquet")
    print(f"graph features: train={train_df.shape}, test={test_df.shape}")
    return train_df, test_df


def raw_feature_columns(df: pd.DataFrame) -> list[str]:
    drop_cols = [
        "is_reference_valid",
        "article_id",
        "ref_id",
        "vector_text_article",
        "vector_text_ref",
        "split",
    ]
    X = df.drop(columns=drop_cols, errors="ignore")
    return X.select_dtypes(include=["number", "bool"]).columns.tolist()


class ProbabilityAdapter:
    """Adapts single-column/3D array output to the format expected by LIME."""
    def __init__(self, model, reshape_to: tuple[int, ...] | None = None):
        self.model = model
        self.reshape_to = reshape_to

    def predict_proba(self, X):
        X_array = np.asarray(X)
        if self.reshape_to is not None and X_array.ndim == 2:
            X_array = X_array.reshape((len(X_array), *self.reshape_to))

        proba = np.asarray(self.model.predict_proba(X_array))
        if proba.ndim == 1:
            proba = np.column_stack([1 - proba, proba])
        elif proba.ndim == 2 and proba.shape[1] == 1:
            positive = proba[:, 0]
            proba = np.column_stack([1 - positive, positive])
        return proba


def prepared_explainability_data(model, train_df: pd.DataFrame, test_df: pd.DataFrame):
    """Prepares X/y consistent with the preprocessing saved inside the model."""
    if hasattr(model, "preprocess"):
        X_train, y_train = model.preprocess(train_df, is_training=False, verbose=False)
        X_test, y_test = model.preprocess(test_df, is_training=False, verbose=False)
    else:
        cols = raw_feature_columns(train_df)
        X_train = train_df[cols].copy()
        X_test = test_df[cols].copy()
        y_train = train_df["is_reference_valid"].copy()
        y_test = test_df["is_reference_valid"].copy()

    reshape_to = None
    if np.asarray(X_train).ndim == 3:
        shape = np.asarray(X_train).shape
        reshape_to = shape[1:]
        X_train = np.asarray(X_train).reshape(shape[0], -1)
        X_test = np.asarray(X_test).reshape(np.asarray(X_test).shape[0], -1)

    if hasattr(model, "article_cols") and getattr(model, "article_cols", None):
        feature_names = list(model.article_cols) + list(model.ref_cols)
    elif isinstance(X_train, pd.DataFrame):
        feature_names = X_train.columns.tolist()
    else:
        feature_names = raw_feature_columns(train_df)
        if len(feature_names) != np.asarray(X_train).shape[1]:
            feature_names = [f"feature_{i:03d}" for i in range(np.asarray(X_train).shape[1])]

    X_train_df = pd.DataFrame(X_train, columns=feature_names, index=train_df.index)
    X_test_df = pd.DataFrame(X_test, columns=feature_names, index=test_df.index)
    y_train = pd.Series(y_train, index=train_df.index, name="is_reference_valid")
    y_test = pd.Series(y_test, index=test_df.index, name="is_reference_valid")

    return X_train_df, X_test_df, y_train, y_test, ProbabilityAdapter(model, reshape_to=reshape_to)


def sample_for_explainability(X: pd.DataFrame, y: pd.Series, max_rows: int = EXPLANATION_SAMPLE_SIZE):
    if max_rows is None or len(X) <= max_rows:
        return X, y

    y = pd.Series(y, index=X.index)
    required_idx = y.groupby(y).head(1).index
    remaining_idx = y.index.difference(required_idx)
    n_remaining = max(max_rows - len(required_idx), 0)

    if n_remaining > 0:
        sampled_idx = remaining_idx.to_series().sample(
            n=min(n_remaining, len(remaining_idx)),
            random_state=RANDOM_STATE,
        ).index
        idx = required_idx.union(sampled_idx)
    else:
        idx = required_idx[:max_rows]

    return X.loc[idx], y.loc[idx]


def tree_model_for_shap(model):
    candidate = getattr(model, "model", model)
    class_name = candidate.__class__.__name__.lower()
    tree_keywords = ["xgb", "lgb", "randomforest", "decisiontree", "gradientboost", "catboost"]
    if any(keyword in class_name for keyword in tree_keywords):
        return candidate
    return None


def run_explainability_block(title: str, models: dict[str, object], train_df: pd.DataFrame, test_df: pd.DataFrame):
    display(Markdown(f"## {title}"))

    if not models:
        print("No models available for this block.")
        return

    for model_name, model in models.items():
        print("" + "=" * 90)
        print(f"Explainability for model: {model_name}")

        try:
            X_train, X_test, y_train, y_test, lime_model = prepared_explainability_data(model, train_df, test_df)
            X_train_lime, y_train_lime = sample_for_explainability(X_train, y_train)
            X_test_lime, y_test_lime = sample_for_explainability(X_test, y_test)
            print(f"Features used: {X_train_lime.shape[1]}")
            print(f"LIME sample train/test: {X_train_lime.shape}, {X_test_lime.shape}")
        except Exception as exc:
            print(f"Data preparation failed: {exc}")
            continue

        try:
            lime_explainer(X_train_lime, X_test_lime, y_test_lime, lime_model)
            print("LIME completed")
        except Exception as exc:
            print(f"LIME failed: {exc}")

        shap_model = tree_model_for_shap(model)
        if shap_model is None:
            print("SHAP skipped: the model is not a tree model supported by TreeExplainer.")
            continue

        try:
            shap_tree_explainer(X_test_lime, shap_model)
            print("SHAP completed")
        except Exception as exc:
            print(f"SHAP skipped/failed: {exc}")

# Initial features

In [ ]:
models_initial = load_all_models(MODEL_DIRS["initial_features"], selected_model="XGB")
train_initial, test_initial = load_graph_split(INITIAL_FEATURES_PATH)

run_explainability_block(
    "Initial features",
    models_initial,
    train_initial,
    test_initial
)

## Textual embedding 128

In [ ]:
models_128 = load_all_models(MODEL_DIRS["embedding_128"], selected_model="Transformer")
train128, test128 = load_textual_split(TEXTUAL_128_PATH)

run_explainability_block(
    "Embedding testuali 128",
    models_128,
    train128,
    test128,
)

## Graph features

In [ ]:
models_graph = load_all_models(MODEL_DIRS["graph_features"], selected_model="XGB")
train_graph, test_graph = load_graph_split(GRAPH_FEATURES_PATH)

run_explainability_block(
    "Graph features",
    models_graph,
    train_graph,
    test_graph,
)

# Mix features

In [ ]:
models_mix = load_all_models(MODEL_DIRS["mix_features"], selected_model="XGB")
train_mix, test_mix = load_graph_split(GRAPH_FEATURES_PATH)

run_explainability_block(
    "Mix features",
    models_mix,
    train_mix,
    test_mix,
)